1. Average sampling of data from all country training sets (1000 each).

In [ ]:
import json
import random
from pathlib import Path
from typing import List, Dict, Any

random.seed(42)


def get_country_dict():
    return {'iran': 'irn',   'israel': 'isr',               'egypt': 'egy',   'saudi arabia': 'sau',
            'turkey': 'tur', 'iraq': 'irq',                 'yemen': 'yem',   'syria': 'syr',
            'jordan': 'jor', 'united arab emirates': 'are', 'lebanon': 'lbn', 'oman': 'omn',
            'kuwait': 'kwt', 'qatar': 'qat',                'bahrain': 'bhr', 'cyprus': 'cyp',
            'palestine': 'pse', }

import json
import random
import os
from typing import Dict, List

def process_json_files(file_paths, samples_per_file: int = 2000):
    
    merged_data = {}
    current_index = 0
    
    for file_country, file_path in file_paths.items():
        
        with open(file_path, 'r', encoding='latin-1') as f:
            data = json.load(f)
        
        available_keys = list(data.keys())
        num_samples = min(samples_per_file, len(available_keys))
        selected_keys = random.sample(available_keys, num_samples)
         
        for key in selected_keys:
            sample_data = data[key].copy()  
            sample_data['country_name'] = file_country
            sample_data['number_country'] = key
            merged_data[str(current_index)] = sample_data
            current_index += 1
    
    return merged_data

def save_merged_data(merged_data: Dict, output_path: str):
    
    with open(output_path, 'w', encoding='latin-1') as f:
        json.dump(merged_data, f, indent=4)

def main():

    file_100_20 = "/ThinkTank-ME/POLECAT-FOR-ME/3_country_dataset/multiexpert/country_max100_min20"
    json_files = {}

    mideast_country_dict = get_country_dict()
    for index_i, content_i in mideast_country_dict.items():
        file_i = f"{file_100_20}/{index_i}_{content_i}/train_object.json"
        json_files[index_i] = file_i
    
    samples_per_file = 1000
    merged_data = process_json_files(json_files, samples_per_file)
    save_merged_data(merged_data, './country_select_max100_min20/train_country_select.json')
    print(f"{len(merged_data)}")

main()

处理完成。总样本数：17000


2. Aggregate the results predicted by 35 experts on a subset of 1 block generate.

In [ ]:

import json
import numpy as np
from pathlib import Path
from typing import Dict, List, Union

def get_country_dict() -> Dict[str, str]:
    return {'iran': 'irn',   'israel': 'isr',               'egypt': 'egy',   'saudi arabia': 'sau',
            'turkey': 'tur', 'iraq': 'irq',                 'yemen': 'yem',   'syria': 'syr',
            'jordan': 'jor', 'united arab emirates': 'are', 'lebanon': 'lbn', 'oman': 'omn',
            'kuwait': 'kwt', 'qatar': 'qat',                'bahrain': 'bhr', 'cyprus': 'cyp',
            'palestine': 'pse', 
            'china': 'chn',     'united states': 'usa',   'russia': 'rus', 'united kingdom': 'gbr', 'france': 'fra',
            'germany': 'deu',   'korea': 'kor',     'japan': 'jpn',  'india': 'ind',     'canada': 'can', 
            'italy': 'ita',     'australia': 'aus', 'spain': 'esp',  'argentina': 'arg', 'brazil': 'bra',
            'indonesia': 'idn', 'mexico': 'mex',    'south africa': 'zaf'}

def process_country_forecasts(base_path: str, beam_size: int = 1) -> Dict:
    """
    处理所有国家的预测结果并整合分析
    
    Args:
        base_path (str): JSON文件所在的基础路径
        beam_size (int): beam search的大小
        
    Returns:
        Dict: 整合后的预测结果
    """
    all_country_dict = get_country_dict()
    train_country_result = {}
    
    for country_name, country_code in all_country_dict.items():
        country_id = country_code.upper()
        file_path = Path(base_path) / f"35A_{country_id}_object/country_select_result.json"
        
        try:
            with open(file_path, "r", encoding='latin-1') as f:
                country_json = json.load(f)
        except FileNotFoundError:
            print(f"警告: 未找到国家 {country_name} 的文件: {file_path}")
            continue
        except json.JSONDecodeError:
            print(f"警告: 国家 {country_name} 的文件格式无效: {file_path}")
            continue
            
        for key, item in country_json.items():
            outputs = item["output"][:beam_size]
            scores = item["scores"][:beam_size]
            
            # 初始化或验证结果字典
            if key not in train_country_result:
                train_country_result[key] = initialize_result_entry(item)
            else:
                assert train_country_result[key]["input"] == item["input"], \
                    f"输入不匹配: key={key}, country={country_name}"
            
            # 处理输出和分数
            process_outputs_and_scores(
                train_country_result[key],
                outputs,
                scores,
                country_name,
                item["target"]
            )
    
    # 处理完所有文件后，为每个结果添加correct_max_country
    for key, item in train_country_result.items():
        add_correct_max_country(item)
    
    return train_country_result

def initialize_result_entry(item: Dict) -> Dict:
    """
    初始化结果字典的入口
    
    Args:
        item (Dict): 原始数据项
        
    Returns:
        Dict: 初始化的结果入口
    """
    result = item.copy()
    result.pop("output")
    result.pop("scores")
    
    result.update({
        "output": {},
        "scores": {},
        "correct_forecast": [],
        "correct_scores": []
    })
    
    return result

def process_outputs_and_scores(
    result: Dict,
    outputs: List[str],
    scores: List[float],
    country_name: str,
    target: str
) -> None:
    """
    处理输出和分数，更新结果字典
    
    Args:
        result (Dict): 要更新的结果字典
        outputs (List[str]): 输出列表
        scores (List[float]): 分数列表
        country_name (str): 国家名称
        target (str): 目标值
    """
    for idx, output in enumerate(outputs):
        clean_output = output.strip().split("<|")[0]
        
        result["output"][country_name] = clean_output
        result["scores"][country_name] = scores[idx]
        
        if target == clean_output:
            result["correct_forecast"].append(country_name)
            result["correct_scores"].append(scores[idx])

def add_correct_max_country(item: Dict) -> None:
    """
    根据correct_scores添加correct_max_country属性
    
    Args:
        item (Dict): 要处理的结果字典项
    """
    if item["correct_scores"]:
        # 找到最大分数的索引
        max_score_index = np.argmax(item["correct_scores"])
        # 设置对应的国家为correct_max_country
        item["correct_max_country"] = item["correct_forecast"][max_score_index]
    else:
        # 如果没有正确预测，使用"None
        item["correct_max_country"] = "None"

def save_results(results: Dict, output_path: str) -> None:
    """
    保存处理结果到JSON文件
    
    Args:
        results (Dict): 处理结果
        output_path (str): 输出文件路径
    """
    with open(output_path, 'w', encoding='latin-1') as f:
        json.dump(results, f, indent=4)

BASE_PATH = "PATH_TO_EXPERT_MODEL_OUTPUTS"
BEAM_SIZE = 1
OUTPUT_PATH = "./country_select_max100_min20/country_forecast_results.json"

# 处理所有国家的预测结果
results = process_country_forecasts(BASE_PATH, BEAM_SIZE)

# 保存结果
save_results(results, OUTPUT_PATH)

3. Aggregate the results predicted by 35 experts on the test set.

In [ ]:
# 合并35个agent在测试集上跑完之后的结果
import json
import numpy as np
from pathlib import Path
from typing import Dict, List, Union

def get_country_dict() -> Dict[str, str]:
    """
    返回国家名称到国家代码的映射字典
    
    Returns:
        Dict[str, str]: 国家名称到ISO代码的映射
    """
    return {'iran': 'irn',   'israel': 'isr',               'egypt': 'egy',   'saudi arabia': 'sau',
            'turkey': 'tur', 'iraq': 'irq',                 'yemen': 'yem',   'syria': 'syr',
            'jordan': 'jor', 'united arab emirates': 'are', 'lebanon': 'lbn', 'oman': 'omn',
            'kuwait': 'kwt', 'qatar': 'qat',                'bahrain': 'bhr', 'cyprus': 'cyp',
            'palestine': 'pse', 
            'china': 'chn',     'united states': 'usa',   'russia': 'rus', 'united kingdom': 'gbr', 'france': 'fra',
            'germany': 'deu',   'korea': 'kor',     'japan': 'jpn',  'india': 'ind',     'canada': 'can', 
            'italy': 'ita',     'australia': 'aus', 'spain': 'esp',  'argentina': 'arg', 'brazil': 'bra',
            'indonesia': 'idn', 'mexico': 'mex',    'south africa': 'zaf'}

def process_country_forecasts(base_path: str, beam_size: int = 1) -> Dict:
    """
    处理所有国家的预测结果并整合分析
    
    Args:
        base_path (str): JSON文件所在的基础路径
        beam_size (int): beam search的大小
        
    Returns:
        Dict: 整合后的预测结果
    """
    all_country_dict = get_country_dict()
    train_country_result = {}
    
    for country_name, country_code in all_country_dict.items():
        country_id = country_code.upper()
        file_path = Path(base_path) / f"35A_{country_id}_object/test_result.json"
        
        try:
            with open(file_path, "r", encoding='latin-1') as f:
                country_json = json.load(f)
        except FileNotFoundError:
            print(f"警告: 未找到国家 {country_name} 的文件: {file_path}")
            continue
        except json.JSONDecodeError:
            print(f"警告: 国家 {country_name} 的文件格式无效: {file_path}")
            continue
            
        for key, item in country_json.items():
            outputs = item["output"][:beam_size]
            scores = item["scores"][:beam_size]
            
            # 初始化或验证结果字典
            if key not in train_country_result:
                train_country_result[key] = initialize_result_entry(item)
            else:
                assert train_country_result[key]["input"] == item["input"], \
                    f"输入不匹配: key={key}, country={country_name}"
            
            # 处理输出和分数
            process_outputs_and_scores(
                train_country_result[key],
                outputs,
                scores,
                country_name,
                item["target"]
            )
    
    # 处理完所有文件后，为每个结果添加correct_max_country
    for key, item in train_country_result.items():
        add_correct_max_country(item)
    
    return train_country_result

def initialize_result_entry(item: Dict) -> Dict:
    """
    初始化结果字典的入口
    
    Args:
        item (Dict): 原始数据项
        
    Returns:
        Dict: 初始化的结果入口
    """
    result = item.copy()
    result.pop("output")
    result.pop("scores")
    
    result.update({
        "output": {},
        "scores": {},
        "correct_forecast": [],
        "correct_scores": []
    })
    
    return result

def process_outputs_and_scores(
    result: Dict,
    outputs: List[str],
    scores: List[float],
    country_name: str,
    target: str
) -> None:
    """
    处理输出和分数，更新结果字典
    
    Args:
        result (Dict): 要更新的结果字典
        outputs (List[str]): 输出列表
        scores (List[float]): 分数列表
        country_name (str): 国家名称
        target (str): 目标值
    """
    for idx, output in enumerate(outputs):
        clean_output = output.strip().split("<|")[0]
        
        result["output"][country_name] = clean_output
        result["scores"][country_name] = scores[idx]
        
        if target == clean_output:
            result["correct_forecast"].append(country_name)
            result["correct_scores"].append(scores[idx])

def add_correct_max_country(item: Dict) -> None:
    """
    根据correct_scores添加correct_max_country属性
    
    Args:
        item (Dict): 要处理的结果字典项
    """
    if item["correct_scores"]:
        # 找到最大分数的索引
        max_score_index = np.argmax(item["correct_scores"])
        # 设置对应的国家为correct_max_country
        item["correct_max_country"] = item["correct_forecast"][max_score_index]
    else:
        # 如果没有正确预测，使用"None
        item["correct_max_country"] = "None"

def save_results(results: Dict, output_path: str) -> None:
    """
    保存处理结果到JSON文件
    
    Args:
        results (Dict): 处理结果
        output_path (str): 输出文件路径
    """
    with open(output_path, 'w', encoding='latin-1') as f:
        json.dump(results, f, indent=4)

BASE_PATH = "PATH_TO_EXPERT_MODEL_OUTPUTS"
BEAM_SIZE = 1
OUTPUT_PATH = "./country_select_max100_min20/country_forecast_results_test.json"

# 处理所有国家的预测结果
results = process_country_forecasts(BASE_PATH, BEAM_SIZE)

# 保存结果
save_results(results, OUTPUT_PATH)

4. Training dataset construction.

In [ ]:
# # 形成训练集

import json
import math
import random
from typing import Dict, List, Union

random.seed(42)

def save_results(results: Dict, output_path: str) -> None:
    """
    保存处理结果到JSON文件
    
    Args:
        results (Dict): 处理结果
        output_path (str): 输出文件路径
    """
    with open(output_path, 'w', encoding='latin-1') as f:
        json.dump(results, f, indent=4)

def get_country_name():
    return {'iran': 'Iran',   'israel': 'Israel',               'egypt': 'Egypt',   'saudi arabia': 'Saudi Arabia',
            'turkey': 'Turkey', 'iraq': 'Iraq',                 'yemen': 'Yemen',   'syria': 'Syria',
            'jordan': 'Jordan', 'united arab emirates': 'United Arab Emirates', 'lebanon': 'Lebanon', 'oman': 'Oman',
            'kuwait': 'Kuwait', 'qatar': 'Qatar',                'bahrain': 'Bahrain', 'cyprus': 'Cyprus',
            'palestine': 'Palestine', 
            'china': 'China',     'united states': 'United States',   'russia': 'Russia', 'united kingdom': 'United Kingdom', 'france': 'France',
            'germany': 'Germany',   'korea': 'Korea',     'japan': 'Japan',  'india': 'India',     'canada': 'Canada', 
            'italy': 'Italy',     'australia': 'Australia', 'spain': 'Spain',  'argentina': 'Argentina', 'brazil': 'Brazil',
            'indonesia': 'Indonesia', 'mexico': 'Mexico',    'south africa': 'South Africa'}

system_prompt = f"Your task is to analyze [Historical Events] and [Current Event to Predict], and then select the most appropriate expert by reasoning from among the candidates to make a prediction." \
                    "\n\n[Candidates] consist of the nationalities of different experts. [Historical Events] consist of a series of timestamped event sets, with each set containing multiple atomic events. The [Current Event to Predict] is presented as a query requiring the identification of the most plausible object, based on the given subject, relationship, and timestamp. " \
                    "Each timestamp is represented as [year-month-day], and atomic events are formatted as triples [subject, relation, object]. " \
                    "For example, [France, REQUEST_meet, Russia] indicates that  'France' (subject) has the relation 'REQUEST_meet' with 'Russia' (object)."\

def get_user_prompt(history, dict_forecast, dict_scores):

    country_name = get_country_name()

    cp = ""
    for (_i, forecast_i), (_ii, socre_i) in zip(dict_forecast.items(), dict_scores.items()):
        assert _i == _ii

        cp = cp + f"\n{country_name[_i]}: [{forecast_i}, {math.exp(socre_i):.2%}]"

    cp = []
    for i, ii in country_name.items():
        cp.append(ii)

    cp = str(cp)
    user_prompt = f"{history}\n\n[Candidates]\n{cp}\n\nPlease use your deep understanding of the predicted events and the broader geopolitical environment, to deliver accurate, contextually informed selections. Only provide the nationality of the expert without additional explanation. If you think that no expert can make a correct prediction, please output 'None'."

    return user_prompt

file_path = "./country_select_max100_min20/country_forecast_results.json"
with open(file_path, "r", encoding='latin-1') as f:
    country_json = json.load(f)

final_country_select_train = {}
country_name = get_country_name()

all_train_number = 0

for index_i, content_i in country_json.items():
    correct_forecast_i = content_i["correct_forecast"]

    if correct_forecast_i != []:

        for correct_forecast_ii in correct_forecast_i:

            final_country_select_train[all_train_number] = {}
            final_country_select_train[all_train_number]["system_prompt"] = system_prompt

            history = content_i["input"].split("\n\nPlease")[0]
            dict_forecast_i =content_i["output"]
            dict_scores_i =content_i["scores"]

            user_prompt_i = get_user_prompt(history, dict_forecast_i, dict_scores_i)

            final_country_select_train[all_train_number]["input"] = user_prompt_i
            final_country_select_train[all_train_number]["target"] = country_name[correct_forecast_ii]

            all_train_number = all_train_number + 1


OUTPUT_PATH = "./country_select_max100_min20/country_select_train_dataset.json"
save_results(final_country_select_train, OUTPUT_PATH)

5. Test dataset construction.

In [ ]:
# 形成测试集

import json
import math
import random
from typing import Dict, List, Union

def save_results(results: Dict, output_path: str) -> None:
    """
    保存处理结果到JSON文件
    
    Args:
        results (Dict): 处理结果
        output_path (str): 输出文件路径
    """
    with open(output_path, 'w', encoding='latin-1') as f:
        json.dump(results, f, indent=4)

def get_country_name():
    return {'iran': 'Iran',   'israel': 'Israel',               'egypt': 'Egypt',   'saudi arabia': 'Saudi Arabia',
            'turkey': 'Turkey', 'iraq': 'Iraq',                 'yemen': 'Yemen',   'syria': 'Syria',
            'jordan': 'Jordan', 'united arab emirates': 'United Arab Emirates', 'lebanon': 'Lebanon', 'oman': 'Oman',
            'kuwait': 'Kuwait', 'qatar': 'Qatar',                'bahrain': 'Bahrain', 'cyprus': 'Cyprus',
            'palestine': 'Palestine', 
            'china': 'China',     'united states': 'United States',   'russia': 'Russia', 'united kingdom': 'United Kingdom', 'france': 'France',
            'germany': 'Germany',   'korea': 'Korea',     'japan': 'Japan',  'india': 'India',     'canada': 'Canada', 
            'italy': 'Italy',     'australia': 'Australia', 'spain': 'Spain',  'argentina': 'Argentina', 'brazil': 'Brazil',
            'indonesia': 'Indonesia', 'mexico': 'Mexico',    'south africa': 'South Africa'}

system_prompt = f"Your task is to analyze [Historical Events] and [Current Event to Predict], and then select the most appropriate expert by reasoning from among the candidates to make a prediction." \
                    "\n\n[Candidates] consist of the nationalities of different experts. [Historical Events] consist of a series of timestamped event sets, with each set containing multiple atomic events. The [Current Event to Predict] is presented as a query requiring the identification of the most plausible object, based on the given subject, relationship, and timestamp. " \
                    "Each timestamp is represented as [year-month-day], and atomic events are formatted as triples [subject, relation, object]. " \
                    "For example, [France, REQUEST_meet, Russia] indicates that  'France' (subject) has the relation 'REQUEST_meet' with 'Russia' (object)."\

def get_user_prompt(history, dict_forecast, dict_scores):

    country_name = get_country_name()

    cp = ""
    for (_i, forecast_i), (_ii, socre_i) in zip(dict_forecast.items(), dict_scores.items()):
        assert _i == _ii

        cp = cp + f"\n{country_name[_i]}: [{forecast_i}, {math.exp(socre_i):.2%}]"

    cp = []
    for i, ii in country_name.items():
        cp.append(ii)

    cp = str(cp)
    user_prompt = f"{history}\n\n[Candidates]\n{cp}\n\nPlease use your deep understanding of the predicted events and the broader geopolitical environment, to deliver accurate, contextually informed selections. Only provide the nationality of the expert without additional explanation."
    return user_prompt

file_path = "./country_select_max100_min20/country_forecast_results_test.json"
with open(file_path, "r", encoding='latin-1') as f:
    country_json = json.load(f)

final_country_select_train = {}
country_name = get_country_name()

for index_i, content_i in country_json.items():
    final_country_select_train[index_i] = {}
    final_country_select_train[index_i]["system_prompt"] = system_prompt

    history = content_i["input"].split("\n\nPlease")[0]
    dict_forecast_i =content_i["output"]
    dict_scores_i =content_i["scores"]

    user_prompt_i = get_user_prompt(history, dict_forecast_i, dict_scores_i)

    final_country_select_train[index_i]["input"] = user_prompt_i
    final_country_select_train[index_i]["target"] = country_name[content_i["correct_max_country"]]

OUTPUT_PATH = "./country_select_max100_min20/country_select_test_dataset.json"
save_results(final_country_select_train, OUTPUT_PATH)